# MongoDB to PostgreSQL Migration
Connect to both DBs and run migrations sequentially.

In [ ]:
import asyncio
import os
import json
import uuid
from datetime import datetime
import motor.motor_asyncio
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession
from sqlalchemy import text
from dotenv import load_dotenv

load_dotenv()
from orchestrator_service.config.application_config import get_config

cfg = get_config()
m_url = f'mongodb://{cfg.mongodb.username}:{cfg.mongodb.password}@{cfg.mongodb.host}:{cfg.mongodb.port}/'
m_client = motor.motor_asyncio.AsyncIOMotorClient(m_url)
m_db = m_client[cfg.mongodb.database]

pg_url = cfg.postgresql.async_url
engine = create_async_engine(pg_url, echo=False)
session_factory = async_sessionmaker(engine, class_=AsyncSession)
print('Connected to both DBs')

In [ ]:
def to_uuid(obj_id) -> str:
    if not obj_id:
        return None
    s = str(obj_id)
    if len(s) == 24:
        return str(uuid.UUID(s + '00000000'))
    return s


## 1. Migrate Users

In [ ]:
async def migrate_users():
    print('Migrating users...')
    cursor = m_db.users.find({})
    count = 0
    async with session_factory() as session:
        async for doc in cursor:
            uid = str(doc['_id'])  # Mezon user ID
            username = doc.get('username')
            display_name = doc.get('display_name')
            avatar_url = doc.get('avatar_url')
            permissions = doc.get('permissions', [])
            created_at = doc.get('created_at')
            updated_at = doc.get('updated_at')

            await session.execute(text('''
                INSERT INTO users (id, username, display_name, avatar_url, permissions, created_at, updated_at)
                VALUES (:id, :uname, :dname, :avatar, :perms::jsonb, :cat, :uat)
                ON CONFLICT (id) DO NOTHING
            '''), {
                'id': uid, 'uname': username, 'dname': display_name, 'avatar': avatar_url,
                'perms': json.dumps(permissions), 'cat': created_at, 'uat': updated_at
            })
            count += 1
        await session.commit()
    print(f'Migrated {count} users.')

await migrate_users()

## 2. Migrate Rooms

In [ ]:
async def migrate_rooms():
    print('Migrating rooms...')
    cursor = m_db.rooms.find({})
    count = 0
    async with session_factory() as session:
        async for doc in cursor:
            uid = to_uuid(doc['_id'])
            mongo_id = str(doc['_id'])
            room_name = doc.get('room_name')
            status = doc.get('status')
            participants = doc.get('participants', [])
            created_at = doc.get('created_at')
            finalized_at = doc.get('finalized_at')
            completed_at = doc.get('completed_at')

            await session.execute(text('''
                INSERT INTO rooms (id, mongo_id, room_name, status, participants, created_at, finalized_at, completed_at)
                VALUES (:id, :mid, :rname, :status, :parts::jsonb, :cat, :fat, :comp)
                ON CONFLICT (id) DO NOTHING
            '''), {
                'id': uid, 'mid': mongo_id, 'rname': room_name, 'status': status,
                'parts': json.dumps(participants), 'cat': created_at, 'fat': finalized_at, 'comp': completed_at
            })
            count += 1
        await session.commit()
    print(f'Migrated {count} rooms.')

await migrate_rooms()

## 3. Migrate Tracks

In [ ]:
async def migrate_tracks():
    print('Migrating tracks...')
    cursor = m_db.tracks.find({})
    count = 0
    async with session_factory() as session:
        async for doc in cursor:
            uid = str(doc['_id'])  # egress_id is already a string
            track_id = doc.get('track_id')
            room_ref_id = to_uuid(doc.get('room_ref_id'))
            pid = doc.get('participant_identity')
            status = doc.get('status')
            chunk_count = doc.get('chunk_count', 0)
            audio_info = doc.get('audio_info')
            error = doc.get('error')
            created_at = doc.get('created_at')
            updated_at = doc.get('updated_at')

            await session.execute(text('''
                INSERT INTO tracks (id, track_id, room_ref_id, participant_identity, status, chunk_count, audio_info, error, created_at, updated_at)
                VALUES (:id, :tid, :rid, :pid, :status, :cc, :audio::jsonb, :error, :cat, :uat)
                ON CONFLICT (id) DO NOTHING
            '''), {
                'id': uid, 'tid': track_id, 'rid': room_ref_id, 'pid': pid, 'status': status,
                'cc': chunk_count, 'audio': json.dumps(audio_info) if audio_info else None,
                'error': error, 'cat': created_at, 'uat': updated_at
            })
            count += 1
        await session.commit()
    print(f'Migrated {count} tracks.')

await migrate_tracks()

## 4. Migrate Chunks

In [ ]:
async def migrate_chunks():
    print('Migrating transcript chunks...')
    cursor = m_db.transcript_chunks.find({})
    count = 0
    async with session_factory() as session:
        async for doc in cursor:
            uid = to_uuid(doc['_id'])
            track_ref_id = str(doc.get('track_ref_id'))
            chunk_index = doc.get('chunk_index')
            start_time = doc.get('start_time')
            end_time = doc.get('end_time')
            item_count = doc.get('item_count')
            segments = doc.get('segments', [])

            await session.execute(text('''
                INSERT INTO transcript_chunks (id, track_ref_id, chunk_index, start_time, end_time, item_count, segments)
                VALUES (:id, :tid, :idx, :st, :et, :ic, :seg::jsonb)
                ON CONFLICT (id) DO NOTHING
            '''), {
                'id': uid, 'tid': track_ref_id, 'idx': chunk_index, 'st': start_time, 'et': end_time,
                'ic': item_count, 'seg': json.dumps(segments)
            })
            count += 1
        await session.commit()
    print(f'Migrated {count} transcript chunks.')

await migrate_chunks()

## 5. Migrate Summary

In [ ]:
async def migrate_summary():
    print('Migrating rooms summary...')
    cursor = m_db.rooms_summary.find({})
    count = 0
    async with session_factory() as session:
        async for doc in cursor:
            uid = to_uuid(doc['_id'])
            room_id = to_uuid(doc.get('room_id'))
            room_name = doc.get('room_name')
            participants = doc.get('participants', [])
            summary_data = doc.get('summary_data')
            full_text = doc.get('full_text')
            messages = doc.get('messages', [])
            total_segments = doc.get('total_segments')
            created_at = doc.get('created_at')

            await session.execute(text('''
                INSERT INTO rooms_summary (id, room_id, room_name, participants, summary_data, full_text, messages, total_segments, created_at)
                VALUES (:id, :rid, :rname, :parts::jsonb, :sdata::jsonb, :ft, :msgs::jsonb, :ts, :cat)
                ON CONFLICT (id) DO NOTHING
            '''), {
                'id': uid, 'rid': room_id, 'rname': room_name, 'parts': json.dumps(participants),
                'sdata': json.dumps(summary_data) if summary_data else None, 'ft': full_text,
                'msgs': json.dumps(messages), 'ts': total_segments, 'cat': created_at
            })
            count += 1
        await session.commit()
    print(f'Migrated {count} summaries.')

await migrate_summary()

## 6. Migrate Metadata Events

In [ ]:
async def migrate_metadata():
    print('Migrating metadata events...')
    cursor = m_db.metadata_events.find({})
    count = 0
    async with session_factory() as session:
        async for doc in cursor:
            uid = to_uuid(doc['_id'])
            event_id = doc.get('event_id')
            event_type = doc.get('event_type')
            room_id = to_uuid(doc.get('room_id'))
            room_name = doc.get('room_name')
            metadata = doc.get('metadata')
            timestamp = doc.get('timestamp')
            created_at = doc.get('created_at')

            await session.execute(text('''
                INSERT INTO metadata_events (id, event_id, event_type, room_id, room_name, metadata, timestamp, created_at)
                VALUES (:id, :eid, :etype, :rid, :rname, :meta::jsonb, :ts, :cat)
                ON CONFLICT (event_id) DO NOTHING
            '''), {
                'id': uid, 'eid': event_id, 'etype': event_type, 'rid': room_id, 'rname': room_name,
                'meta': json.dumps(metadata) if metadata else None, 'ts': str(timestamp) if timestamp else None, 'cat': created_at
            })
            count += 1
        await session.commit()
    print(f'Migrated {count} metadata events.')

await migrate_metadata()

## 7. Cleanup

In [ ]:
await engine.dispose()
m_client.close()
print('Connections closed.')